# Workshop 2 - Power BI Dataset Readiness (exercise)

![Power BI mockup](../../assets/images/powerbi_report_mockup.png)

## Scenario

RetailHub is ready to hand the Gold model to Power BI. You decide which objects support the summary page and drill-through page, choose Import or DirectQuery for this dataset, verify the incremental-refresh window and complete the BI contract and cost guardrails.

## Objectives

After completing this workshop you will be able to:

- choose the correct Gold object for a report page,
- justify Import mode vs DirectQuery for a concrete dataset,
- test incremental-refresh date boundaries,
- fill a BI contract for a handoff,
- define cost guardrails for a Power BI report on Databricks.

## Prerequisites

Before starting, complete:

1. `setup/00_setup.ipynb`
2. `notebooks/demo/day1_02_gold_modeling_for_powerbi.ipynb`
3. `notebooks/demo/day2_01_powerbi_semantic_model.ipynb`

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this workshop.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before starting the tasks.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream demo notebook has not created the required Gold objects.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day1_02_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly) and notebooks/demo/day2_01_powerbi_semantic_model.ipynb (for v_fact_sales_incremental)")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day1_02_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly) and notebooks/demo/day2_01_powerbi_semantic_model.ipynb (for v_fact_sales_incremental)\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## Success Criteria

You are done when:

1. Summary-page source is chosen and justified (row count, grain known).
2. Drill-through source is chosen and justified (date boundaries known).
3. Import vs DirectQuery is decided for THIS dataset, with reasoning - not
   just copied from the Day 2 demo.
4. You ran your own boundary test against `gold.v_fact_sales_incremental`
   and it returns only the requested half-open window.
5. The BI contract (`docs/templates/bi-contract.md` shape) is filled for
   this workshop's choices.
6. The cost guardrails checklist (`docs/templates/cost-awareness-checklist.md`
   shape) is filled for this report.

## Tasks

1. Wybierz source dla summary page.
2. Wybierz source dla drill-through.
3. Zdefiniuj Import vs DirectQuery dla tego datasetu.
4. Zweryfikuj incremental refresh (twoj wlasny boundary test).
5. Wypelnij BI contract.
6. Wypelnij cost guardrails.

## Task 1 - wybierz source dla summary page

Inspect the monthly aggregate. Expected output: month range, row count and
total revenue.

In [ ]:
# TODO: inspect the monthly aggregate and note month range, row count, revenue.
spark.sql(f"SELECT * FROM {GOLD}.fact_sales_dashboard_monthly LIMIT 20").show()

# TODO: write your decision below.
# Decision: summary page source = ?
# Reason: ?

## Task 2 - wybierz source dla drill-through

Inspect the incremental view. Expected output: date range and detail row
count.

In [ ]:
# TODO: inspect the incremental view and note its date range and row count.
spark.sql(f"SELECT MIN(order_date), MAX(order_date), COUNT(*) FROM {GOLD}.v_fact_sales_incremental").show()

# TODO: write your decision below.
# Decision: drill-through source = ?
# Reason: ?

## Task 3 - zdefiniuj Import vs DirectQuery (ten dataset)

Apply the Day 2 semantic model decision framework to THIS report - not a generic answer.
Fill in the table for your two chosen sources.

| Question | Your answer for this report |
|---|---|
| Does any visual need sub-hour freshness? | |
| Is data volume small enough for Import? | |
| Are there concurrent users hitting the same visuals? | |
| Is there a genuine operational/live use case? | |

# TODO: write your Import vs DirectQuery decision and reasoning here.

## Task 4 - zweryfikuj incremental refresh

Pick a different `RangeStart`/`RangeEnd` window than the Day 2 demo example and
prove the half-open contract holds (no row exactly on `RangeEnd`).

Expected output: rows only between your chosen `RangeStart` and before
`RangeEnd`, and zero rows reported as exactly on `RangeEnd`.

In [ ]:
# TODO: pick your own range_start / range_end (different from the Day 2 demo example).
range_start = "2025-04-01"
range_end = "2025-07-01"

# TODO: apply the half-open RangeStart / RangeEnd predicate.
spark.sql(f"""
SELECT COUNT(*) AS rows_in_window
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '{range_start}'
  -- TODO: add the RangeEnd predicate (must be < not <=)
""").show()

# TODO: add a second query that counts rows exactly on range_end and assert it is 0.

## Task 5 - wypelnij BI contract

Fill the `docs/templates/bi-contract.md` shape for this workshop's dataset.

| Area | Decision |
|---|---|
| Summary source | |
| Detail/drill-through source | |
| Grain (summary) | |
| Grain (detail) | |
| Baseline mode | |
| Live option | |
| Refresh pattern | |
| Cost guardrail | |
| Refresh owner | |
| Business owner | |
| Technical owner | |

<!-- TODO: fill every row of the BI contract table above. -->

## Task 6 - wypelnij cost guardrails

Fill the `docs/templates/cost-awareness-checklist.md` shape for this report.

| Area | Question | Decision |
|---|---|---|
| Warehouse size | What size is enough for the demo/report? | |
| Auto-stop | How long should idle compute stay warm? | |
| Import mode | Can refresh be scheduled instead of live queries? | |
| DirectQuery | Which visuals can trigger expensive SQL? | |
| Aggregates | Which Gold aggregates reduce scan size? | |
| Monitoring | Where do we inspect query and billing usage? | |

<!-- TODO: fill every row of the cost guardrails table above. -->

## Bonus

- add a separate source for a live operational page,
- decide which visuals should use aggregate vs detail,
- list three fields that must become slicers,
- write one DAX measure and decide whether it belongs in Power BI or Gold.

## Summary

In this workshop you decide how Power BI should consume Databricks Gold objects: summary source, drill-through source, Import or DirectQuery, incremental-refresh boundaries, BI contract and cost guardrails.

The expected output is a defensible handoff package, not only a working query.